In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/docai/bin/python


In [2]:
import importlib

packages = [
    "requests",
    "dotenv",
    "langgraph",
    "pdfplumber",
    "docx",
    "PIL",
    "pydantic",
]

for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f"✅ {pkg} is installed")
    except ImportError:
        print(f"❌ {pkg} NOT installed")


✅ requests is installed
✅ dotenv is installed
✅ langgraph is installed
✅ pdfplumber is installed
✅ docx is installed
✅ PIL is installed
✅ pydantic is installed


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("✅ API key loaded successfully")
else:
    print("❌ API key NOT found")

✅ API key loaded successfully


In [6]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={API_KEY}"

payload = {
    "contents": [
        {"parts": [{"text": "Say hello in one sentence"}]}
    ]
}

response = requests.post(url, json=payload)

print(response.status_code)
print(response.json())


200
{'candidates': [{'content': {'parts': [{'text': 'Hello there!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}], 'usageMetadata': {'promptTokenCount': 5, 'candidatesTokenCount': 3, 'totalTokenCount': 38, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 5}], 'thoughtsTokenCount': 30, 'serviceTier': 'standard'}, 'modelVersion': 'gemini-2.5-flash', 'responseId': 'ppSyasy7CaX4juMPwL_QiQs'}


In [7]:
from typing import TypedDict, Optional, Dict, Any, List

# defining the state that will be passed to each node, each node will be configured to modify only their respective fields
class DocumentState(TypedDict):
    # --- Input ---
    file_name: str
    file_type: Optional[str]          # pdf, docx, image
    raw_bytes: bytes                  # original uploaded file
    extracted_text: Optional[str]     # text extracted from file

    # --- Classification ---
    document_type: Optional[str]      # invoice, purchase_order, receipt, other
    classification_confidence: Optional[float]

    # --- Extraction ---
    extracted_fields: Optional[Dict[str, Any]] # date, ref no, location

    # --- Validation ---
    validation_errors: Optional[List[str]] # to be defined
    is_valid: Optional[bool]

    # --- Final Output ---
    final_output: Optional[Dict[str, Any]]


In [20]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

def call_gemini(prompt):
    
    url = (
        "https://generativelanguage.googleapis.com/"
        f"v1/models/gemini-2.5-flash:generateContent?key={API_KEY}"
    )

    payload = {
        "contents": [
            {
                "parts": [
                    {"text": prompt}
                ]
            }
        ]
    }

    response = requests.post(url, json=payload)

    if response.status_code != 200:
        raise Exception(
            f"Gemini API error {response.status_code}: "
            f"{response.text}"
        )

    result = response.json()

    if "candidates" not in result:
        raise Exception(f"Unexpected Gemini response: {result}")

    return result["candidates"][0]["content"]["parts"][0]["text"]

In [10]:
import pdfplumber
from docx import Document
from PIL import Image
import io


def ingest_node(state: DocumentState) -> DocumentState:
    print("Running ingest node")

    file_name = state["file_name"]
    raw_bytes = state["raw_bytes"]

    # Detect file type
    if file_name.endswith(".pdf"):
        state["file_type"] = "pdf"

        with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
            text = "\n".join(page.extract_text() or "" for page in pdf.pages)

        state["extracted_text"] = text

    elif file_name.endswith(".docx"):
        state["file_type"] = "docx"

        doc = Document(io.BytesIO(raw_bytes))
        text = "\n".join(p.text for p in doc.paragraphs)

        state["extracted_text"] = text

    elif file_name.endswith((".png", ".jpg", ".jpeg")):
        state["file_type"] = "image"

        # placeholder for OCR later
        state["extracted_text"] = "image text extraction not implemented"

    else:
        state["file_type"] = "unknown"
        state["extracted_text"] = ""

    return state

In [22]:
import json

def classify_node(state: DocumentState) -> DocumentState:
    print("Running classify node")

    text = state["extracted_text"][:5000]

    prompt = f"""
You are a document classification agent.

Classify the document into exactly one of these categories:

- invoice
- purchase_order
- receipt
- other

Return ONLY valid JSON in this exact format:

{{
    "document_type": "invoice",
    "confidence": 0.95
}}

Rules:
- document_type must be exactly one of:
  invoice, purchase_order, receipt, other
- confidence must be a number between 0 and 1.
- Do not include markdown.
- Do not include any explanation.

Document:
{text}
"""

    response = call_gemini(prompt)

    try:
        result = json.loads(response)
    except json.JSONDecodeError:
        raise ValueError(
            f"Gemini returned invalid classification JSON:\n{response}"
        )

    state["document_type"] = result["document_type"]
    state["classification_confidence"] = float(result["confidence"])

    return state

In [24]:
def extract_node(state: DocumentState) -> DocumentState:
    print("Running extract node")

    doc_type = state["document_type"]
    text = state["extracted_text"][:6000]

    if doc_type == "invoice":

        prompt = f"""
You are an invoice extraction agent.

Extract the following fields from the invoice:

- invoice_number
- vendor
- customer
- invoice_date
- due_date
- subtotal
- tax
- total_amount

Return ONLY valid JSON in exactly this structure:

{{
    "invoice_number": null,
    "vendor": null,
    "customer": null,
    "invoice_date": null,
    "due_date": null,
    "subtotal": null,
    "tax": null,
    "total_amount": null
}}

Rules:
- Use null if a field cannot be found.
- Do not guess missing information.
- Numeric money fields must be numbers, without currency symbols.
- Keep dates as strings.
- Do not include markdown.
- Do not include explanations.

Document:
{text}
"""

        response = call_gemini(prompt)

        try:
            data = json.loads(response)
        except json.JSONDecodeError:
            raise ValueError(
                f"Gemini returned invalid extraction JSON:\n{response}"
            )

        state["extracted_fields"] = data

    else:
        state["extracted_fields"] = {}

    return state


In [13]:
def validate_node(state: DocumentState) -> DocumentState:
    print("Running validate node")

    data = state["extracted_fields"]

    try:
        InvoiceSchema(**data)

        state["is_valid"] = True
        state["validation_errors"] = []

    except Exception as e:

        state["is_valid"] = False
        state["validation_errors"] = [str(e)]

    return state

In [14]:
from langgraph.graph import StateGraph, END
# Create graph builder
builder = StateGraph(DocumentState)

# Add nodes
builder.add_node("ingest", ingest_node)
builder.add_node("classify", classify_node)
builder.add_node("extract", extract_node)
builder.add_node("validate", validate_node)
# builder.add_node("publish", publish_node)
# builder.add_node("review", review_node)

# Set entry point
builder.set_entry_point("ingest")

# Linear edges
builder.add_edge("ingest", "classify")
builder.add_edge("classify", "extract")
builder.add_edge("extract", "validate")


In [15]:
# def route_after_validation(state: DocumentState):
#     if state["is_valid"]:
#         return "publish"
#     return "review"


# builder.add_conditional_edges(
#     "validate",
#     route_after_validation,
#     {
#         "publish": "publish",
#         "review": "review"
#     }
# )

# # End states
# builder.add_edge("publish", END)
# builder.add_edge("review", END)

In [16]:
app = builder.compile()

In [17]:
with open("sample.pdf", "rb") as f:
    pdf_bytes = f.read()

print(type(pdf_bytes))
print(len(pdf_bytes))

<class 'bytes'>
43627


In [18]:
initial_state = {
    "file_name": "sample.pdf",
    "file_type": None,
    "raw_bytes": pdf_bytes,
    "extracted_text": None,
    "document_type": None,
    "classification_confidence": None,
    "extracted_fields": None,
    "validation_errors": None,
    "is_valid": None,
    "final_output": None
}

# result = app.invoke(initial_state)

# print(result)

result = ingest_node(initial_state)

print(result["file_type"])
print(result["extracted_text"][:1000])


Running ingest node
pdf
Invoice
Invoice Number INV-3337
From:
DEMO - Sliced Invoices Order Number 12345
Suite 5A-1204 Invoice Date January 25, 2016
123 Somewhere Street
Due Date January 31, 2016
Your City AZ 12345
Total Due $93.50
admin@slicedinvoices.com
To:
Test Business
123 Somewhere St
d
Melbourne, VIC 3000
test@test.com
i
Hrs/Qty Service Rate/Price Adjust Sub Total
a
Web Design
1.00 $85.00 0.00% $85.00
This is a sample description...
P
Sub Total $85.00
Tax $8.50
Total $93.50
ANZ Bank
ACC # 1234 1234
BSB # 4321 432
Payment is due within 30 days from date of invoice. Late payment is subject to fees of 5% per month.
Thanks for choosing DEMO - Sliced Invoices | admin@slicedinvoices.com
Page 1/1


In [21]:
print(call_gemini("What type of document is an invoice? Answer in one word."))

Bill


In [25]:
state = ingest_node(initial_state)
state = classify_node(state)
state = extract_node(state)

print(state["document_type"])
print(state["classification_confidence"])
print(state["extracted_fields"])

Running ingest node
Running classify node
Running extract node
invoice
0.99
{'invoice_number': 'INV-3337', 'vendor': 'DEMO - Sliced Invoices', 'customer': 'Test Business', 'invoice_date': 'January 25, 2016', 'due_date': 'January 31, 2016', 'subtotal': 85.0, 'tax': 8.5, 'total_amount': 93.5}
